In [2]:
from arcgis.gis import GIS
import pandas as pd
import getpass
from credentials import username, password

# inventory_group_items.py
# Connect to ArcGIS Online and get an inventory of all items in a group.
# Saves results to CSV and returns a pandas DataFrame.


def connect_agol():
    """
    Connect to ArcGIS Online.
    If you provide a username, you'll be prompted for a password.
    Leave username blank to connect anonymously (only works for public groups/items).
    """
    user = username
    if not user:
        return GIS()  # anonymous / built-in authentication flow
    pwd = password
    return GIS("https://www.arcgis.com", user, pwd)

def get_group_inventory(gis, group_id, save_csv=True, csv_path=None, max_items=10000):
    """
    Return a DataFrame with an inventory of items in the given group.
    csv_path: optional path to save CSV. If None and save_csv True, uses group_<id>_inventory.csv
    max_items: maximum number of items to fetch (set high for large groups).
    """
    group = gis.groups.get(group_id)
    if group is None:
        raise ValueError(f"Group not found: {group_id}")

    # try group.content(), fallback to a search by group id
    try:
        items = group.content(max_items=max_items)
    except Exception:
        # fallback: search for items that belong to the group
        # query using group:"<group_id>" can work as a fallback
        items = gis.content.search(query=f'group:"{group_id}"', max_items=max_items)

    records = []
    for it in items:
        records.append({
            "id": it.id,
            "title": it.title,
            "type": it.type,
            "owner": it.owner,
            "created": pd.to_datetime(it.created, unit='ms'),
            "modified": pd.to_datetime(it.modified, unit='ms'),
            "size": getattr(it, "size", None),
            "url": getattr(it, "url", None),
            "access": getattr(it, "access", None),
            "tags": ",".join(it.tags) if it.tags else "",
            "snippet": getattr(it, "snippet", None),
            "description": getattr(it, "description", None),
            "typeKeywords": ",".join(it.typeKeywords) if getattr(it, "typeKeywords", None) else ""
        })

    df = pd.DataFrame.from_records(records, columns=[
        "id","title","type","owner","created","modified","size","url","access",
        "tags","snippet","description","typeKeywords"
    ])

    if save_csv:
        if csv_path is None:
            csv_path = f"group_{group_id}_inventory.csv"
        df.to_csv(csv_path, index=False)
        print(f"Saved inventory to {csv_path}")

    return df

if __name__ == "__main__":
    gis = connect_agol()
    group_id = "b094a38a8ebe4017b8f41cc7b0f6be22"
    df = get_group_inventory(gis, group_id)
    print(df)

Saved inventory to group_b094a38a8ebe4017b8f41cc7b0f6be22_inventory.csv
                                   id  \
0    f270332bbdef455fbcdf3ca32c038958   
1    3f769b482c744f55ba4473fcd97af311   
2    d4636681923643a480aea7a919f27f3b   
3    f08cec2e848044ef935c09e522a3f019   
4    a343330454104f0fa35dc27a7e46db69   
..                                ...   
365  cdccd8a8ce664979b073792e6cdb951e   
366  dadfc10da21f43e69c888eaadf5198d4   
367  42eba1b7e4c5454cbe4149179d29153f   
368  06c6bf6a2ba24e0ab2529cd99a9eb130   
369  5788c5cfca82418fa65054bbccd899ac   

                                          title                 type  \
0        Monthly Vessel Traffic Density - Cargo        Image Service   
1      Monthly Vessel Traffic Density - Fishing        Image Service   
2     Monthly Vessel Traffic Density - Military        Image Service   
3        Monthly Vessel Traffic Density - Other        Image Service   
4    Monthly Vessel Traffic Density - Passenger        Image Service   
.. 

In [6]:
# get all content items in the folder "US Vessel Traffic App" for the current user
folder_name = "US Vessel Traffic App"
user = gis.users.me

# find the folder id by title
folder_id = '5e5302cdd1794945b89bf48f2fa06a1a'

# list items in the folder
items = user.items(folder=folder_id, max_items=10000)

# build a DataFrame similar to the existing df
records = []
for it in items:
    records.append({
        "id": it.id,
        "title": it.title,
        "type": it.type,
        "owner": it.owner,
        "created": pd.to_datetime(it.created, unit="ms") if getattr(it, "created", None) is not None else None,
        "modified": pd.to_datetime(it.modified, unit="ms") if getattr(it, "modified", None) is not None else None,
        "size": getattr(it, "size", None),
        "url": getattr(it, "url", None),
        "access": getattr(it, "access", None),
        "tags": ",".join(it.tags) if it.tags else "",
        "snippet": getattr(it, "snippet", None),
        "description": getattr(it, "description", None),
        "typeKeywords": ",".join(it.typeKeywords) if getattr(it, "typeKeywords", None) else ""
    })

folder_items_df = pd.DataFrame.from_records(records, columns=[
    "id","title","type","owner","created","modified","size","url","access",
    "tags","snippet","description","typeKeywords"
])

folder_items_df

,id,title,type,owner,created,modified,size,url,access,tags,snippet,description,typeKeywords
0,ced9f549b29042a0924ea3cc69f47bbe,AIS_App_Source_Data,Microsoft Excel,esri_oceans,2020-12-23 14:11:52,2020-12-23 18:40:08,18512,None,private,"AIS,Vessel Traffic,Shipping",None,None,"Data,Document,Microsoft Excel"
1,61390198f1ef40d5af6321d51fcbb518,AIS_App_Source_Data,Feature Service,esri_oceans,2020-12-23 14:13:11,2021-12-07 20:04:49,73728,https://services.arcgis.com/P3ePLMYs2RVChkJx/a...,public,"AIS,Vessel Traffic,Shipping,unlisted",None,None,"ArcGIS Server,Data,Feature Access,Feature Serv..."
2,6d700b63a9244867ae92010918d6d18e,Chart_Index,Service Definition,esri_oceans,2021-01-07 18:41:20,2021-01-07 18:41:21,1757623,,private,"AIS,Vessel Traffic,Chart Index",Chart Index for Vessel Tracker App,,".sd,ArcGIS,Metadata,Service Definition,Online"
3,4ed0b58d07f8481db248faeccfdb7714,Chart_Index,Feature Service,esri_oceans,2021-01-07 18:41:24,2021-10-12 02:27:13,884736,https://services.arcgis.com/P3ePLMYs2RVChkJx/a...,public,"AIS,Vessel Traffic,Chart Index,unlisted",Chart Index for Vessel Tracker App,,"ArcGIS Server,Data,Feature Access,Feature Serv..."
4,31ecee1eb9df45278e5df8a56797b684,AIS File Download Size,CSV,esri_oceans,2021-01-13 02:36:12,2021-01-13 02:38:17,30560,None,private,"AIS,Vessel Traffic",None,None,CSV
...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,409ab7e1b94447c68619eedf5c6bc9c2,US_Vessel_Traffic_2024_10_optimized,Vector Tile Service,esri_oceans,2025-10-30 03:14:57,2025-10-30 03:16:13,1979285918,https://tiles.arcgis.com/tiles/P3ePLMYs2RVChkJ...,public,"AIS,Vessel Traffic,U.S. Vessel Traffic",Monthly AIS Vessel Traffic Vector Tiles for US...,<div style='text-align:Left;font-size:12pt'><d...,"Data,Service,Vector Tile Service,ArcGIS Server..."
371,e5676547a840479299c691ed978aed27,US_Vessel_Traffic_2024_11_optimized,Vector Tile Package,esri_oceans,2025-10-30 03:38:53,2025-10-30 03:38:57,1744206653,,shared,"AIS,Vessel Traffic,U.S. Vessel Traffic",Monthly AIS Vessel Traffic Vector Tiles for US...,<div style='text-align:Left;font-size:12pt'><d...,"ArcGIS Pro,Data,Vector Tile Package,vtpk"
372,c7e8b4631b9947af9eb5b650b33df876,US_Vessel_Traffic_2024_11_optimized,Vector Tile Service,esri_oceans,2025-10-30 03:38:59,2025-10-30 03:40:18,1744025264,https://tiles.arcgis.com/tiles/P3ePLMYs2RVChkJ...,public,"AIS,Vessel Traffic,U.S. Vessel Traffic",Monthly AIS Vessel Traffic Vector Tiles for US...,<div style='text-align:Left;font-size:12pt'><d...,"Data,Service,Vector Tile Service,ArcGIS Server..."
373,0783fa1066534792b8a1ac541065192f,US_Vessel_Traffic_2024_12_optimized,Vector Tile Package,esri_oceans,2025-10-30 12:41:37,2025-10-30 12:41:41,1648715011,,shared,"AIS,Vessel Traffic,U.S. Vessel Traffic",Monthly AIS Vessel Traffic Vector Tiles for US...,<div style='text-align:Left;font-size:12pt'><d...,"ArcGIS Pro,Data,Vector Tile Package,vtpk"


In [9]:
# save folder_items_df to CSV (uses existing folder_name and folder_items_df)
csv_filename = f"{folder_name.replace(' ', '_')}_folder_items.csv"
folder_items_df.to_csv(csv_filename, index=False)
print(f"Saved {len(folder_items_df)} rows to {csv_filename}")

Saved 375 rows to US_Vessel_Traffic_App_folder_items.csv
